# Live runs on Mistral

Mistral has a batch queue, but it is entitled separately from the rest of the
api and the account does not reach it, so this model is generated live: one
request at a time, appending each reply as it arrives, with a running cost.

At $0.15 and $0.60 per million that costs about $0.79 against $0.40 batched, so
the difference is under forty pence and not worth chasing an entitlement for.

The pass is cut into five parts, so that a long run is checkpointed rather than
all or nothing and progress is legible in whole chunks. Each part writes the raw
responses in the shape a batch job would have returned, is read straight into
the results, and reports the same lines the batch notebooks report. The parts
are then joined into one file and removed, leaving a single record of what the
provider returned.

Interrupting is safe at any point. Every reply is written as it arrives, and
re-running a part asks only for what that part still lacks.

In [1]:
# Import the libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [2]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

/Users/rinlobachevskii/Desktop/Git/Thesis


In [3]:
# Import the pipeline
%load_ext autoreload
%autoreload 2

import backends
import flags
import run
import settings
import utils

needs = {'flags': ['apply', 'report', 'flags_of'],
         'run': ['generate_part', 'join_parts', 'read_batch', 'part_path',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path'],
         'backends': ['USAGE', 'spent', 'record_usage', 'call_api'],
         'settings': ['MODELS', 'GENERATION', 'BATCHES_DIR']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

Scripts are current


## The model

In [4]:
MODEL = 'mistral-small-2603'
PARTS = 5

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print(f'Model      {MODEL} on {spec["provider"]}')
print(f'Billed at  ${spec["price"]["input"]}/M input, '
      f'${spec["price"]["output"]}/M output, batch queue not entitled')
print(f'Cap        {settings.GENERATION["max_tokens"]} tokens, '
      f'temperature {settings.GENERATION["temperature"]}')
print(f'Collected  {have:,} of {wanted:,}, in {PARTS} parts of '
      f'{-(-wanted // PARTS):,}')
print(f'Key found  {bool(utils.api_key(spec["provider"]))}')

Model      mistral-small-2603 on mistral
Billed at  $0.15/M input, $0.6/M output, batch queue not entitled
Cap        4096 tokens, temperature 1.0
Collected  0 of 4,320, in 5 parts of 864
Key found  True


## Rerunning

`FRESH` moves an earlier pass to `results/superseded/` and asks for every prompt
again. Leave it false to finish a pass that stopped part way, which is the
normal case here since a live run of four thousand calls will not always
complete in one sitting.

In [5]:
FRESH = True

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

Nothing collected yet, so nothing to set aside


## What it should cost

Run `test_batch.py mistral-small-2603` first to replace the guess. There is no
batch rate to halve here, so this figure is what you pay.

In [6]:
OUTPUT_TOKENS = 300          # Replace with what test_batch.py measures
INPUT_TOKENS = 24

left = wanted - have
price = spec['price']
cost = (left * INPUT_TOKENS * price['input']
        + left * OUTPUT_TOKENS * price['output']) / 1e6
print(f'{left:,} calls outstanding at {OUTPUT_TOKENS} output tokens each')
print(f'  Cost  ${cost:,.2f}, about ${cost / PARTS:,.2f} a part')

4,320 calls outstanding at 300 output tokens each
  Cost  $0.79, about $0.16 a part


## Generate, one part at a time

Each part is its own cell, so a part that finishes is banked whatever happens to
the next one. Run them in order, or re-run any single one: a part already
collected reports nothing outstanding rather than being asked for again.

Requests go out several at a time. A live call spends nearly all of its time
waiting rather than sending, so this finishes in a fraction of the time and
costs exactly the same.

In [7]:
# Define once, then run each part below. Re-running a part asks only for what
# that part still lacks, so an interrupted part costs nothing but its own time.
totals = {'read': 0, 'failed': 0, 'truncated': 0, 'repeated': 0,
          'blocked': 0, 'input': 0, 'output': 0, 'cost': 0.0}


def run_part(part):
    path, asked, failures = run.generate_part(MODEL, part, PARTS)
    if not asked:
        print(f'Part {part} of {PARTS}: nothing outstanding')
        return

    # counted from the ingest rather than the generation, so that a response
    # recorded on the way out and again on the way in is not billed twice
    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, path)
    usage, cost = dict(backends.USAGE), backends.spent(MODEL)
    for name, value in [('read', read), ('failed', failed),
                        ('truncated', truncated), ('repeated', repeated),
                        ('blocked', blocked), ('input', usage['input']),
                        ('output', usage['output']), ('cost', cost)]:
        totals[name] += value

    print(f'\nPart {part} of {PARTS}')
    print(f'Read {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output')
    print(f'Cost: ${cost:,.2f}')
    print(f'Output tokens a reply: {usage["output"] / max(read - failed, 1):.0f}')


print(f'{PARTS} parts of {-(-wanted // PARTS):,}, '
      f'{utils.WORKERS} requests in flight at a time')

5 parts of 864, 12 requests in flight at a time


In [8]:
run_part(1)

  mistral-small-2603 part 1  48 of 864, 2650 an hour, 0.3 hours left, 0 failed
     $0.0033 spent, $0.06 projected for this pass, 6,711 tokens
  mistral-small-2603 part 1  96 of 864, 2661 an hour, 0.3 hours left, 0 failed
     $0.0071 spent, $0.06 projected for this pass, 14,167 tokens
  mistral-small-2603 part 1  144 of 864, 2642 an hour, 0.3 hours left, 0 failed
     $0.0105 spent, $0.06 projected for this pass, 21,086 tokens
  mistral-small-2603 part 1  192 of 864, 2563 an hour, 0.3 hours left, 0 failed
     $0.0184 spent, $0.08 projected for this pass, 35,520 tokens
  mistral-small-2603 part 1  216 of 864, 2348 an hour, 0.3 hours left, 0 failed
     $0.0398 spent, $0.16 projected for this pass, 71,875 tokens
  mistral-small-2603 part 1  264 of 864, 2389 an hour, 0.3 hours left, 0 failed
     $0.0438 spent, $0.14 projected for this pass, 79,730 tokens
  mistral-small-2603 part 1  312 of 864, 2414 an hour, 0.2 hours left, 0 failed
     $0.0498 spent, $0.14 projected for this pass, 90

In [ ]:
run_part(2)

  mistral-small-2603 part 2  48 of 864, 2708 an hour, 0.3 hours left, 0 failed
     $0.1379 spent, $2.48 projected for this pass, 252,981 tokens
  mistral-small-2603 part 2  96 of 864, 2706 an hour, 0.3 hours left, 0 failed
     $0.1400 spent, $1.26 projected for this pass, 257,643 tokens
  mistral-small-2603 part 2  144 of 864, 2617 an hour, 0.3 hours left, 0 failed
     $0.1484 spent, $0.89 projected for this pass, 272,806 tokens
  mistral-small-2603 part 2  192 of 864, 2551 an hour, 0.3 hours left, 0 failed
     $0.1529 spent, $0.69 projected for this pass, 281,422 tokens
  mistral-small-2603 part 2  240 of 864, 2554 an hour, 0.2 hours left, 0 failed
     $0.1593 spent, $0.57 projected for this pass, 293,304 tokens
  mistral-small-2603 part 2  288 of 864, 2546 an hour, 0.2 hours left, 0 failed
     $0.1646 spent, $0.49 projected for this pass, 303,402 tokens
  mistral-small-2603 part 2  336 of 864, 2569 an hour, 0.2 hours left, 0 failed
     $0.1679 spent, $0.43 projected for this p

In [ ]:
run_part(3)

In [ ]:
run_part(4)

In [ ]:
run_part(5)

## Join and total

Run once every part is done. The parts are joined into one file and removed,
leaving a single record of what the provider returned.

In [ ]:
joined, lines = run.join_parts(MODEL, PARTS)
print(f'Joined {lines:,} responses into {joined.name}, part files removed')

print(f'\nAll parts')
print(f'Read {totals["read"]:,} replies, {totals["failed"]} failed, '
      f'{totals["truncated"]} truncated, {totals["repeated"]:,} already had, '
      f'{totals["blocked"]} blocked')
print(f'Tokens: {totals["input"]:,} input, {totals["output"]:,} output')
print(f'Cost: ${totals["cost"]:,.2f}')
print(f'Output tokens a reply: '
      f'{totals["output"] / max(totals["read"] - totals["failed"], 1):.0f}')

## Check what arrived

In [ ]:
# Bring the two flags up to date from the raw provider file, then report.
# Safe to re-run: it recomputes from data/batches/ rather than accumulating.
flags.apply(MODEL)
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    marked = lambda name: collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['True', 'true']) \
        | ~collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['', 'False', 'false', 'nan'])
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored, '
          f'{int(marked("blocked").sum())} blocked, '
          f'{int(marked("truncated").sum())} truncated')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))